<a href="https://colab.research.google.com/github/SUZUB/ATU/blob/main/ATU_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# Cell 1: Infrastructure & Cloud Sync
# ==============================================================================
import os
from google.colab import drive

# 1. Public Git clone / pull
if not os.path.exists('ATU') and not os.path.exists('/content/ATU'):
    !git clone https://github.com/SUZUB/ATU.git
if os.path.exists('/content/ATU'):
    %cd /content/ATU
elif os.path.exists('ATU'):
    %cd ATU
# Force-sync to the latest GitHub version. /content/ATU is a disposable mirror:
# edit code locally and push; any edits made here are discarded (listed first).
!git fetch -q origin && git status --short && git reset -q --hard origin/main
!git log -1 --format="Repo synced to commit %h (%cd): %s" --date=short

# Install project requirements
!pip install -q -r requirements.txt

# yfinance >= 1.8 requires curl_cffi >= 0.17 (yfinance's "chrome" profile resolves to
# chrome152). Upgrade both together; an older curl_cffi still loaded in this runtime
# is what raises ImpersonateError('Impersonating chrome152 is not supported').
!pip install -q -U "yfinance>=1.8.0" "curl_cffi>=0.17.0"

import sys, subprocess, importlib.metadata as md
stale = [
    pkg for pkg in ("curl_cffi", "yfinance")
    if pkg in sys.modules and getattr(sys.modules[pkg], "__version__", None) != md.version(pkg)
]
if stale:
    print(f"Outdated {stale} already loaded in this runtime -> restarting. Re-run from Cell 1 afterwards.")
    os.kill(os.getpid(), 9)  # Colab reconnects automatically with the upgraded packages

# Smoke test in a fresh process, using the same session logic as the pipeline
# (market_data_processor.get_yfinance_session picks a browser profile curl_cffi supports)
probe = subprocess.run(
    [sys.executable, "-c",
     "import yfinance as yf, curl_cffi, market_data_processor as m; "
     "d = m._yf_download(tickers='AAPL', period='5d', interval='5m', progress=False); "
     "print(f'yfinance {yf.__version__} | curl_cffi {curl_cffi.__version__} | "
     "profile {m._YF_SESSION_STATE.get(\"profile\")} | AAPL 5m rows: {len(d)}')"],
    capture_output=True, text=True,
)
print(probe.stdout.strip() or f"yfinance smoke test FAILED:
{probe.stderr.strip()[-1500:]}")

# 2. Google Drive Mount
drive.mount('/content/drive')

# 3. Persistent Drive Workspace & Subdirectory Creation
DRIVE_WORKSPACE = '/content/drive/MyDrive/ATU_Workspace'
os.makedirs(DRIVE_WORKSPACE, exist_ok=True)

DATASET_PATH = os.path.join(DRIVE_WORKSPACE, 'datasets')
MODEL_CHECKPOINTS = os.path.join(DRIVE_WORKSPACE, 'checkpoints')
RESULTS_PATH = os.path.join(DRIVE_WORKSPACE, 'results')

os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs(MODEL_CHECKPOINTS, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

print("\nInfrastructure & Cloud Sync complete.")
print(f"Project Directory: {os.getcwd()}")
print(f"Drive Workspace:   {DRIVE_WORKSPACE}")

In [ ]:
# ==============================================================================
# Cell 2: Imports & Setup
# ==============================================================================
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# Ensure ATU root directory is in sys.path
atu_root = Path.cwd()
if str(atu_root) not in sys.path:
    sys.path.insert(0, str(atu_root))

# Drop cached project modules so code pulled in Cell 1 is used without a runtime restart
_project_modules = ("market_data_processor", "sentiment_processor", "constrained_trading_env", "train_and_backtest")
for _name in [m for m in sys.modules if m in _project_modules or m == "src" or m.startswith("src.")]:
    del sys.modules[_name]

# Undo any yf.download monkey-patch left over from an earlier cell (re-running such a
# patch makes it wrap itself -> RecursionError). The pipeline does not need one.
import yfinance as yf
import yfinance.multi
yf.download = yfinance.multi.download

# Shared pipeline (identical logic to `python train_and_backtest.py`)
from src.train import build_live_state_vector, run_experiment

# Device Auto-detection (FinBERT runs on GPU when available; SB3 MLP agents train on CPU)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Imports & Setup complete.")
print(f"PyTorch Version:  {torch.__version__}")
print(f"Active Device:    {DEVICE.upper()}")
if torch.cuda.is_available():
    print(f"GPU Hardware:     {torch.cuda.get_device_name(0)}")

In [ ]:
# ==============================================================================
# Cell 3: Real Market Data + Real News Sentiment -> final_state_vector
# ==============================================================================
# - Real 5-minute candles from yfinance (last 60 days). Raises on failure; no synthetic data.
# - The news step is optional: if it fails, the cell continues with neutral sentiment.
# - Real headlines: Google News RSS (searched in 3-day windows) + yfinance news, cached in
#   <workspace>/datasets/news_cache_<TICKER>.csv so later runs only fetch new days.
# - FinBERT scores, dated in the exchange timezone, lagged 1 day (Tue news -> Wed candles).
# - Session rules: NSE/BSE (.NS/.BO) 09:15-15:30 IST, square-off 15:15;
#                  US 09:30-16:00 ET, square-off 15:45.
TICKER = "AAPL"  # e.g. "RELIANCE.NS"
USE_NEWS_SENTIMENT = True  # False = price data only (skips news + FinBERT; sentiment = 0.0)
DRIVE_WORKSPACE = globals().get("DRIVE_WORKSPACE", "/content/drive/MyDrive/ATU_Workspace")

data = build_live_state_vector(
    ticker=TICKER,
    period="60d",
    data_dir=os.path.join(DRIVE_WORKSPACE, "datasets"),
    device=DEVICE,
    use_news=USE_NEWS_SENTIMENT,
)
final_state_vector = data.final_state_vector
print(f"\nEnv Session Rules: {data.env_kwargs}")
display(final_state_vector.head(3))

In [ ]:
# ==============================================================================
# Cell 4: Multi-Seed Ensemble RL (PPO + A2C) & Backtest
# ==============================================================================
# - One PPO + A2C ensemble per seed, trained on the same 80% and tested on the same 20%.
#   A real edge shows up in most seeds; a single good seed can be luck.
# - Rewards are VecNormalize'd for training only (clip 10); stats saved per agent.
# - Checkpoints resume from <Drive>/checkpoints/seed_<n>/ (incompatible -> fresh model).
# - Backtest runs on a raw env. Per-seed reports, daily P&L and plots are saved to
#   <Drive>/results/seed_<n>/; the cross-seed summary to <Drive>/results/seed_summary.txt.
SEEDS = [0, 1, 2]
CHUNKS = 10
STEPS_PER_CHUNK = 50_000

assert "data" in globals(), "Run Cell 3 first."
experiment = run_experiment(
    data,
    workspace=DRIVE_WORKSPACE,
    seeds=SEEDS,
    chunks=CHUNKS,
    steps_per_chunk=STEPS_PER_CHUNK,
    show_plot=True,
)
print("
" + experiment.report)
